# Three systems, one duty, three rungs

> **Demonstration only:** frozen synthetic data, not evidence about any real decision. A rung is how a conclusion was reached, not a confidence score.

The next cell imports each shipped system and evaluates the same duty through `check_conformance`. The table keeps the result fields that matter: verdict, rung, basis, and the evidence summary.

In [1]:
from dataclasses import replace

from reasonsmith.examples import neural_scorer, probabilistic_scorer, symbolic_rules
from reasonsmith.report import check_conformance
from reasonsmith.spec import load_pack

pack = load_pack("ecoa")
requirement = pack.get_requirement("ecoa_reg_b_1002_9_b_2_specific_reasons")
one_duty = replace(pack, id="ecoa:specific-reasons", requirements=(requirement,))
reports = {}
for name, factory in {
    "neural": neural_scorer.system_under_test,
    "probabilistic": probabilistic_scorer.system_under_test,
    "symbolic": symbolic_rules.system_under_test,
}.items():
    reports[name] = check_conformance(factory(), one_duty)

print("system | duty | verdict | rung | basis | evidence")
for name, report in reports.items():
    item = report.to_dict()["results"][0]
    print(f"{name} | {item['requirement_id']} | {item['verdict']} | {item['strength']} | "
          f"{item['basis']} | {item['evidence_summary'][:100]}...")


system | duty | verdict | rung | basis | evidence
neural | ecoa_reg_b_1002_9_b_2_specific_reasons | satisfied | observed | behavioural | Observed over 3 decision(s): state monitor for 'present(artifact_logs_reason_explanation) -> ( prese...
probabilistic | ecoa_reg_b_1002_9_b_2_specific_reasons | satisfied | probed | behavioural | Probed: no counterexample to 'present(artifact_logs_reason_explanation) -> ( present(provenance_mode...
symbolic | ecoa_reg_b_1002_9_b_2_specific_reasons | satisfied | proved | behavioural | Proved for all inputs: formal solver verified requirement 'present(artifact_logs_reason_explanation)...


`observed`, `probed`, and `proved` are distinct evidence contracts. Now ask the neural log for the shipped reason-deletion duty: it has no inference artefact, so the honest answer is an unattainable refusal with its reason, not a weaker pass.

In [2]:
artifact_requirement = pack.get_requirement(
    "ecoa_reg_b_1002_9_b_2_principal_reasons_complete"
)
artifact_duty = replace(pack, id="ecoa:principal-reasons", requirements=(artifact_requirement,))
refusal_report = check_conformance(neural_scorer.system_under_test(), artifact_duty)
refusal = refusal_report.to_dict()["results"][0]
print({
    "duty": refusal["requirement_id"],
    "verdict": refusal["verdict"],
    "outcome": refusal["outcome"],
    "strength": refusal["strength"],
    "signals_missing": refusal["signals_missing"],
    "reason": refusal["evidence_summary"],
})


{'duty': 'ecoa_reg_b_1002_9_b_2_principal_reasons_complete', 'verdict': 'inconclusive', 'outcome': 'unattainable', 'strength': 'unattainable', 'signals_missing': ['artifact_logs_deleted_reason_count'], 'reason': 'Unattainable as built: the system declares no capability to emit artifact_logs_deleted_reason_count, so no amount of testing can discharge this requirement. Determined from declared capabilities alone; the system was not executed.'}


The refusal is a usable result: the log-only system cannot expose the artefact needed by this duty, and no amount of replay or extra log rows changes that. See [`docs/theory/08-evidence.md`](../docs/theory/08-evidence.md) for the authoritative ladder and refusal semantics.